# Project 2: Bayesian Constraints on Dark Energy with Pantheon+

**PHYS690: Computational Methods for Physics Research**  
**Student name(s):** _replace this text_  
**Private repository URL:** _replace this text_

This project continues the Type Ia supernova Hubble diagram from Assignment 2. You will use the public Pantheon+ standardized magnitudes to compare flat $\Lambda\mathrm{CDM}$ with a time-varying dark-energy model, then use MCMC to describe the correlated parameter uncertainties. The goal is to interpret the posterior, not only find the parameter vector with the lowest χ².

## Purpose and deliverables

Complete this guided notebook and independently complete `scripts/run_pantheon_inference.py`. The script must run from a clean Python process and reproduce the final saved figures, posterior samples, parameter summaries, and chain diagnostics. Do not rely on variables defined only in this notebook.

You will compare a baseline flat $\Lambda\mathrm{CDM}$ model with flat $w_0w_a\mathrm{CDM}$. The extra dark-energy parameter directions are correlated and the supernova-only constraints can be broad and asymmetric. MCMC makes those joint and marginalized distributions visible. A lower best-fit χ² for the more flexible model is expected; by itself it does not show that the extra parameters are required. This project does not calculate a Bayes factor.

## Physical model

Type Ia supernovae are standardized candles. For the flat $w_0w_a\mathrm{CDM}$ parameterization,

$$w(a)=w_0+w_a(1-a), \qquad a=(1+z)^{-1}.$$

The dimensionless expansion rate is

$$E(z)^2=\Omega_m(1+z)^3+(1-\Omega_m)(1+z)^{3(1+w_0+w_a)}\exp\left[-3w_a\frac{z}{1+z}\right].$$

For fixed $H_0$, the luminosity distance and standardized apparent magnitude model are

$$d_L(z)=(1+z)\frac{c}{H_0}\int_0^z\frac{dz'}{E(z')}, \qquad m_{b,\mathrm{model}}=5\log_{10}\left(\frac{d_L}{\mathrm{Mpc}}\right)+25+M_B.$$

Here $M_B$ is a nuisance magnitude intercept. Supernova apparent magnitudes alone do not independently determine $H_0$ and the absolute-magnitude calibration; fixing $H_0$ and fitting $M_B$ preserves that degeneracy instead of claiming an independent $H_0$ measurement. For flat $\Lambda\mathrm{CDM}$, set $w_0=-1$ and $w_a=0$.

## Part 1: Load the public data and covariance

Use the official PantheonPlusSH0ES DataRelease files linked below. The table has 1701 entries; its covariance file begins with the matrix dimension followed by the matrix elements in sequence. Select supernova rows with `IS_CALIBRATOR == 0` and `zHD > 0.01`, then select the corresponding rows and columns from the full covariance matrix. Use `m_b_corr` as the standardized apparent magnitude. Do not use `m_b_corr_err_DIAG` as a replacement for the covariance: the official release labels diagonal errors for visualization only.

References for file names, columns, and covariance format are linked at the end of the notebook.

In [ ]:
%matplotlib inline
from pathlib import Path
from urllib.request import urlretrieve

import emcee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.constants import c
from scipy.linalg import cholesky, solve_triangular
from scipy.optimize import minimize

plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": True})
if Path.cwd().name == "notebooks":
    repo_root = Path.cwd().parent
else:
    repo_root = Path.cwd()
data_dir = repo_root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)
DATA_URL = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
COV_URL = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES_STAT%2BSYS.cov"
data_path = data_dir / "Pantheon+SH0ES.dat"
cov_path = data_dir / "Pantheon+SH0ES_STAT+SYS.cov"
# TODO: Download each file only when it is absent. Then read the table with
# whitespace separation. Read the covariance text, parse its first value N,
# reshape the remaining N*N values, and verify dimensions agree with the table.
# Tip: np.fromstring(text, sep=" ") handles whitespace-separated numbers.
raise NotImplementedError("Load the Pantheon+ table and covariance matrix")

In [ ]:
# TODO: Inspect the columns, missing values, row count, and ranges of zHD,
# m_b_corr, and IS_CALIBRATOR. Plot the full redshift distribution.
# Create the boolean selection mask (non-calibrator and zHD > 0.01).
# Use np.ix_(mask, mask) to select the matching covariance submatrix.
# Report how many entries remain and check that the selected covariance is
# symmetric and positive definite (for example, attempt a Cholesky factorization).
selected = ...

### Data inspection prompt

Explain the selection and why Cepheid-calibrator rows are excluded from this supernova-only fit. Compare the diagonal of the selected full covariance with any diagonal error column available for plotting. Identify at least one reason the off-diagonal terms matter when many surveys and systematics contribute.

## Part 2: Visualize the Hubble diagram and covariance

Make a Hubble diagram using the selected redshifts and standardized magnitudes. Error bars from the covariance diagonal may be shown for visualization, but state that fitting uses the full matrix. Also plot the correlation matrix $R_{ij}=C_{ij}/\sqrt{C_{ii}C_{jj}}$ (or a clear subset if plotting all pixels is not useful).

Save final figures from the independent script to `figures/pantheon_hubble_diagram.png` and `figures/pantheon_covariance.png`.

In [ ]:
# TODO: Build the correlation matrix and make the two labeled figures.
# Show the data with diagonal error bars for visual context only.
# Save the figures to the paths specified above when running the script.
...

## Part 3: Implement the expansion history and magnitude model

Implement $E(z)$ and evaluate the luminosity-distance integral. A fixed Gauss-Legendre quadrature is one efficient option: map quadrature nodes from $[-1,1]$ to $[0,z_i]$ for each supernova, evaluate $1/E(z')$, and sum with the corresponding weights. You can use `numpy.polynomial.legendre.leggauss`. Confirm that $d_L$ increases with redshift and that all model magnitudes are finite over the allowed parameter range.

Use parameters `[Omega_m, w0, wa, M_B]`. Suggested broad bounds are $0.05<\Omega_m<0.6$, $-2.5<w_0<-0.3$, $-5<w_a<5$, and $-20<M_B<-18$. These are modeling choices: report them and later assess whether posterior mass presses against a boundary.

In [ ]:
H0 = 70.0  # km s^-1 Mpc^-1; held fixed, absorbed with calibration by M_B
C_KM_S = c / 1000.0
PARAMETER_NAMES = ["Omega_m", "w0", "wa", "M_B"]
BOUNDS = np.array([[0.05, 0.60], [-2.5, -0.3], [-5.0, 5.0], [-20.0, -18.0]])

def e_of_z(z, omega_m, w0, wa):
    # TODO: Implement the flat w0waCDM expression from the model section.
    raise NotImplementedError

def apparent_magnitude_model(z, parameters, n_quad=32):
    # TODO: Use Gauss-Legendre quadrature for integral_0^z dz'/E(z').
    # Convert d_L to Mpc, then return distance modulus + M_B.
    raise NotImplementedError

## Part 4: Correlated Gaussian likelihood and fit comparison

Let $\mathbf m_{\rm obs}$ be the fixed observed standardized magnitudes and $\mathbf r=\mathbf m_{\rm obs}-\mathbf m_{\rm model}(\boldsymbol{\theta})$. With the full covariance $C$,

$$\log L=-\frac12\left[\mathbf r^{\mathsf T}C^{-1}\mathbf r+\log|C|+N\log(2\pi)\right].$$

Since $C$ is fixed, the last two terms are constant for parameter estimation; they still matter if comparing normalized likelihood values across different data/covariance definitions. Do not calculate $C^{-1}$ explicitly. If $C=LL^{\mathsf T}$, solve $L\mathbf y=\mathbf r$ and compute $\chi^2=\mathbf y^{\mathsf T}\mathbf y$.

Fit $\Lambda\mathrm{CDM}$ with $w_0=-1,w_a=0$ and the four-parameter $w_0w_a\mathrm{CDM}$ model. Use bounded `scipy.optimize.minimize` and several starting points if needed. Report best-fit parameters, $\chi^2$, degrees of freedom, and $\Delta$AIC as a descriptive complexity-penalized comparison. Do not interpret a lower $\chi^2$ alone as evidence that time-varying dark energy is established.

In [ ]:
def whitened_residual(parameters, z, magnitude, covariance_cholesky):
    # TODO: Evaluate the model, form residuals, and solve L y = residual.
    raise NotImplementedError

def chi_square(parameters, z, magnitude, covariance_cholesky):
    # TODO: Return the squared norm of the whitened residual vector.
    raise NotImplementedError

# TODO: Fit the nested LambdaCDM parameter vector [Omega_m, M_B] and the
# time-varying parameter vector [Omega_m, w0, wa, M_B]. Use the same selected
# observations and covariance. Construct a compact comparison DataFrame.
fit_lcdm = ...
fit_w0wa = ...

### Fit and model-comparison prompt

What changes when the extra dark-energy freedom is introduced? Explain why χ² cannot increase at the best fit for a nested, more flexible model, and why that fact alone does not establish that the additional parameters are physically needed. What does AIC add, and what does it leave out? This project does not calculate the Bayesian evidence or a Bayes factor.

## Part 5: Bayes theorem, priors, and the emcee target

For parameters $\boldsymbol{\theta}=(\Omega_m,w_0,w_a,M_B)$ and fixed data $D_{\rm obs}$, Bayes' theorem is

$$p(\boldsymbol{\theta}\mid D_{\rm obs})=\frac{p(D_{\rm obs}\mid\boldsymbol{\theta})p(\boldsymbol{\theta})}{p(D_{\rm obs})}.$$

Use a uniform prior within the stated bounds and zero outside. Thus the log posterior, up to an additive constant, is $-\chi^2/2$ inside the prior support and $-\infty$ outside. The evidence $p(D_{\rm obs})$ is constant with respect to $\boldsymbol{\theta}$ for this model and dataset, so it is not needed when sampling this posterior.

Lecture 18 uses the production package emcee to explore a posterior defined by the same log-probability function. emcee uses an affine-invariant ensemble of walkers. You do not write the sampler's accept/reject loop: provide a function that returns one log-posterior value for one parameter vector, initialize an ensemble of walkers inside the prior, then call emcee.EnsembleSampler. Use 48 walkers for the four model parameters.

Follow the Lecture 18 sequence: run burn-in; keep the returned final walker state; reset the sampler so stored burn-in samples are cleared; then run and retain production steps from that State. Resetting clears stored samples but does not return walkers to their starting positions.

In [ ]:
def log_prior(parameters):
    # TODO: Return 0 inside BOUNDS and -np.inf otherwise.
    raise NotImplementedError

def log_posterior(parameters, z, magnitude, covariance_cholesky):
    # TODO: Add the log prior and the correlated Gaussian log likelihood.
    # emcee will pass (z, magnitude, covariance_cholesky) through args=.
    # Return one scalar log probability for one parameter vector.
    raise NotImplementedError

def run_emcee_ensemble(initial_positions, z, magnitude, covariance_cholesky,
                       burnin_steps, production_steps, seed):
    # TODO: Adapt the production pattern from Lecture 18:
    # 1. Set a reproducible NumPy random seed for emcee.
    # 2. Create emcee.EnsembleSampler(n_walkers, n_parameters, log_posterior,
    #    args=(z, magnitude, covariance_cholesky)).
    # 3. Call run_mcmc(initial_positions, burnin_steps) and save its returned State.
    # 4. Call sampler.reset() to clear stored burn-in draws.
    # 5. Call run_mcmc(burnin_state, production_steps) and return sampler.
    raise NotImplementedError

## Part 6: Initialize ensembles and diagnose emcee sampling

Run four independent ensembles, each with 48 walkers for the four-parameter posterior. Initialize each ensemble in a small, full-rank cloud around the fitted point; the local covariance estimate from the least-squares fit is a useful scale for that cloud. Check that every initial walker has a finite log posterior inside the prior bounds.

For each ensemble, follow Lecture 18: run burn-in, retain its returned final State, reset the sampler's stored chain, and run production from that State. The production chain returned by get_chain() has shape (steps, walkers, parameters). The sampler's acceptance_fraction has one value per walker. Use get_autocorr_time(tol=0) to estimate each parameter's integrated autocorrelation time; tol=0 asks emcee to return the estimate without the usual chain-length warning threshold.

Walkers within one ensemble interact, so do not call them independent chains. For split R-hat, compare the same walker trace across the four independently initialized ensemble runs. Estimate effective sample size from the number of ensemble steps and walkers divided by integrated autocorrelation time. Review traces, acceptance, autocorrelation, ESS, and split R-hat before reporting intervals; a sampler running successfully is not itself proof of convergence.

In [ ]:
N_WALKERS = 48
N_ENSEMBLES = 4
burnin_steps = 500
production_steps = 2000
samplers = []

# TODO: Estimate a local covariance from the least-squares Jacobian and use
# its Cholesky factor to scale a small initial walker cloud around fit_w0wa.x.
# For each independently seeded ensemble:
# - create 48 finite starting positions with shape (48, 4);
# - call run_emcee_ensemble(..., burnin_steps, production_steps, seed);
# - append the returned sampler to samplers.
#
# Then make production_chains with shape (4, production_steps, 48, 4) by
# stacking each sampler.get_chain(). Inspect mean acceptance per ensemble.
# Get integrated autocorrelation times with sampler.get_autocorr_time(tol=0).
production_chains = ...
acceptance = ...
tau_by_ensemble = ...

def split_rhat(chains_by_run):
    # TODO: Use one matching walker trace from each independent ensemble run.
    # Input shape: (n_independent_runs, n_steps, n_parameters).
    raise NotImplementedError

# TODO: Calculate split R-hat, combined approximate ESS, plot the four
# independent walker traces, and save figures/pantheon_mcmc_diagnostics.png.
diagnostics = ...

### Diagnostic decision

State whether the independent ensemble traces and diagnostics support the interval estimates you report. Distinguish posterior width from Monte Carlo error: the posterior describes uncertainty conditional on the data, model, and prior; finite-chain error describes how accurately the computation estimates that posterior. If emcee's integrated autocorrelation time is a substantial fraction of production length, run more production steps and reassess before summarizing.

## Part 7: Marginal and joint posterior interpretation

Use only post-burn-in production states. Flatten the step and walker axes after burn-in has been removed; keep the four independent ensemble runs independent until diagnostics are reviewed. Report median and central 68% and 95% posterior intervals. Make a corner-style figure showing one-dimensional marginal distributions and two-dimensional joint structure for $\Omega_m,w_0,w_a$; include the nuisance intercept in the summary table. Compare posterior medians with the best-fit point and, if you calculated a local covariance approximation, explain what it misses.

A central 68% posterior interval contains 68% of posterior probability under the specified model and prior. It is not a frequentist confidence interval. Asymmetric intervals can arise when the likelihood is curved, bounded, or weakly informative in one direction.

### Prior sensitivity

The broad prior uses $-5<w_a<5$. Compare it with a narrower uniform prior $-2<w_a<2$, holding other priors fixed. You may rerun ensembles with the narrower prior or, if the broad-prior ensembles adequately sample the full range, restrict those draws to $|w_a|<2$. Because both priors are constant within their support, this is the conditional broad-posterior sample corresponding to the narrower prior. Report how many draws remain and how the marginal intervals change. If the ensembles do not adequately sample the broad tails, rerun the sensitivity analysis instead of relying on post hoc filtering.

In [ ]:
# TODO: For each sampler, use sampler.get_chain(flat=True) to obtain its
# production draws (burn-in was cleared with sampler.reset()). Combine the
# independent runs only after reviewing diagnostics.
posterior_draws = ...
posterior_summary = ...

# TODO: Compare LambdaCDM point constraints with w0waCDM marginalized constraints.
# Save figures/pantheon_posterior.png and outputs/pantheon_parameter_summary.csv.

## Part 8: Posterior predictive check

For posterior draws $\boldsymbol{\theta}^{(s)}$, generate replicated magnitude vectors from the multivariate Gaussian observation model,

$$\widetilde{\mathbf m}^{(s)}=\mathbf m_{\rm model}(\boldsymbol{\theta}^{(s)})+L\mathbf z, \qquad \mathbf z\sim\mathcal N(\mathbf 0,I), \quad C=LL^{\mathsf T}.$$

Choose and explain a discrepancy, such as a covariance-weighted residual sum or binned residual trend with redshift. Compare the observed discrepancy to replicated values. A posterior predictive tail fraction is a model-checking diagnostic, not a classical goodness-of-fit p-value. It can expose a failure in the chosen summary, but cannot rule out every unmodeled systematic or alternative cosmology.

In [ ]:
# TODO: Generate posterior predictive replicate(s) using the full covariance.
# Compare a preselected discrepancy for observed and replicated data.
# Plot the observed Hubble diagram, posterior median model, and a predictive
# band or a clear binned-residual diagnostic. Save the figure.
predictive_results = ...

## Part 9: Final reproducible workflow

Complete scripts/run_pantheon_inference.py independently of notebook state. It should download the official data if missing; apply and record the same row selection; use the full covariance submatrix; fit both cosmologies; run four independent emcee ensembles using 48 walkers each; separate burn-in from production with sampler.reset(); diagnose the production samples; assess prior sensitivity; and save posterior samples, summary tables, and final figures.

Run from the repository root:

python scripts/run_pantheon_inference.py --steps 2000 --warmup 500 --seed 2202

Here steps is the number of production steps per walker in each ensemble; warmup is burn-in steps per walker. Extend production if autocorrelation and convergence diagnostics indicate that these settings are insufficient. Then restart the notebook kernel and run all cells. Explain any small numerical differences between the notebook and script.

## Final discussion

Report the $\Lambda\mathrm{CDM}$ and $w_0w_a\mathrm{CDM}$ parameter estimates, uncertainty conventions, chain diagnostics, and model-comparison statistic. Explain the strongest posterior correlations and whether the marginalized $w_a$ constraint is symmetric. State what the data support and what they do not establish. Address the fixed-$H_0$/absolute-magnitude degeneracy, full-covariance use, prior dependence, and omitted analysis systematics.

_Your final discussion here._

## References

- Pantheon+ DataRelease repository: https://github.com/PantheonPlusSH0ES/DataRelease
- Pantheon+ distance and covariance file notes: https://github.com/PantheonPlusSH0ES/DataRelease/tree/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR
- Brout et al. (2022), The Pantheon+ Analysis: Cosmological Constraints: https://arxiv.org/abs/2202.04077
- Scolnic et al. (2022), The Pantheon+ Analysis: The Full Dataset and Light-Curve Release: https://arxiv.org/abs/2112.03863
- Course lectures 17–19: https://github.com/jrstevenjlab/gradcompphys/tree/main/lectures
- Lecture 17, Bayesian inference: https://github.com/jrstevenjlab/gradcompphys/blob/main/lectures/lecture17_11_5_26_bayesian_inference.ipynb
- Lecture 18, emcee production sampling: https://github.com/jrstevenjlab/gradcompphys/blob/main/lectures/lecture18_11_10_26_mcmc.ipynb
- Lecture 19, autocorrelation, ESS, and model checking: https://github.com/jrstevenjlab/gradcompphys/blob/main/lectures/lecture19_11_12_26_model_checking_flows.ipynb